# A hand-written training loop: `expval` and `jacobian`

`fit` in [`07a`](07a_native_classifier.ipynb) and [`07b`](07b_readouts_and_losses.ipynb) does the whole training: it runs the circuits, computes the loss and moves the parameters. Sometimes the loss or the loop has to be written outside `fit` instead: a loss `fit` does not have, a model with classical parts around it, or another training framework. For that `polypus.qml` has two primitives that compute no loss at all:

- `qml.expval(model, x, params)` returns the expectation of every readout observable for every row of `x`.
- `qml.jacobian(model, x, params)` returns their derivatives with respect to the parameters, computed exactly with the parameter-shift rule.

With them, this notebook writes a gradient descent by hand.

## Data and model

There are six samples with two features each: the first three around small angles with target `+1` and the last three around large angles with target `−1`. The model is the one from [`07a`](07a_native_classifier.ipynb), with a readout of `⟨Z₀⟩` and no decision, so its output is the expectation itself. The parameters are now an ordinary NumPy array, here a random start:

In [ ]:
import numpy as np
from polypus import qml

X = np.array([[0.2, 0.1], [0.4, 0.3], [0.1, 0.5], [2.7, 2.9], [3.0, 2.6], [2.8, 3.1]])
Y = np.array([1.0, 1.0, 1.0, -1.0, -1.0, -1.0])

model = qml.Model(
    qml.Encoder(2).angle_encoder("ry"),
    qml.Ansatz(2).real_amplitudes(1),
    qml.Readout([qml.Z(0)]),
)
params = np.random.default_rng(7).uniform(-np.pi, np.pi, model.num_params)
print(model.num_params, "parameters,", model.num_observables, "observable")

## `expval`: the model's outputs

`expval` returns an array with one row per sample and one column per observable, here 6 × 1. It takes the same execution arguments as `TrainedModel.run`: `exact=True` computes the exact expectations, while with shots each value carries sampling noise:

In [ ]:
values = qml.expval(model, X, params, exact=True)
print(values.shape)
print(np.round(values[:, 0], 3))
print(np.round(qml.expval(model, X, params, shots=1000, seed=3)[:, 0], 3))

## `jacobian`: how the outputs change with each parameter

`jacobian` adds one dimension, the parameters: entry `[i, k, j]` is how much observable `k` of row `i` changes when parameter `j` moves. The parameter-shift rule gets it from two circuits per parameter, with its angle shifted up and down, and it is exact, not an approximation. A finite difference with a very small step, computed with `expval`, gives the same numbers for parameter 0:

In [ ]:
by_params = qml.jacobian(model, X, params, exact=True)
print(by_params.shape)

h = 1e-4
shift = np.zeros_like(params)
shift[0] = h
finite_difference = (
    qml.expval(model, X, params + shift, exact=True)
    - qml.expval(model, X, params - shift, exact=True)
) / (2 * h)
print(np.round(by_params[:, 0, 0], 6))
print(np.round(finite_difference[:, 0], 6))

## The training loop

The loss is a squared error, `mean((⟨Z₀⟩ − y)²)`. Its gradient follows from the chain rule: `2·mean((⟨Z₀⟩ − y)·∂⟨Z₀⟩/∂θ)`, which needs exactly the two arrays above. Each step computes both and moves the parameters against the gradient, with a learning rate of 0.3:

In [ ]:
import matplotlib.pyplot as plt

losses = []
for step in range(40):
    values = qml.expval(model, X, params, exact=True)
    losses.append(float(np.mean((values[:, 0] - Y) ** 2)))
    by_params = qml.jacobian(model, X, params, exact=True)
    # chain rule: d loss / d theta = 2 * mean((<Z0> - y) * d<Z0>/d theta)
    gradient = 2 * (values[:, 0] - Y) @ by_params[:, 0, :] / len(Y)
    params = params - 0.3 * gradient

print("loss:", round(losses[0], 4), "->", round(losses[-1], 4))
print(np.round(qml.expval(model, X, params, exact=True)[:, 0], 3))

plt.figure(figsize=(5, 3))
plt.plot(losses)
plt.xlabel("step")
plt.ylabel("squared error")
plt.show()

After 40 steps the expectations sit near the targets, `+1` for the first three samples and `−1` for the last three.

## Derivatives with respect to the inputs

`wrt="inputs"` differentiates with respect to the features of each sample instead of the parameters, only for angle and IQP encoders. It shows how sensitive the trained model is to each feature, and it is what lets a classical layer placed before the model learn too, as in the next notebook. `wrt="both"` returns the two Jacobians at once:

In [ ]:
by_inputs = qml.jacobian(model, X, params, wrt="inputs", exact=True)
print(by_inputs.shape)
print(np.round(by_inputs[:, 0, :], 3))

## Summary

This notebook computed a native model's outputs with `qml.expval` and their exact derivatives with `qml.jacobian`, checked them against a finite difference, trained the model with a gradient descent of its own on a squared error, and differentiated it with respect to its inputs with `wrt="inputs"`.

The next notebook uses these same two primitives inside PyTorch, to put a native model in a hybrid quantum-classical network.

## Next step

Continue with: [`07d_pytorch_hybrid.ipynb`](07d_pytorch_hybrid.ipynb).